# jitmo tool-picker fine-tune (self-contained)

Same as `finetune_router.ipynb`, but the training data is embedded below (no Drive, no upload) and the finished model is downloaded by the browser. Runtime > Change runtime type > GPU, then Run all.

**Never train on `testset/heldout.txt`.** It is not used here.

In [ ]:
#@title 1. Settings
BASE_MODEL = "unsloth/gemma-3-270m-it"  #@param ["unsloth/gemma-3-270m-it", "unsloth/functiongemma-270m-it", "Qwen/Qwen3.5-0.8B", "unsloth/gemma-3-1b-it"]
EPOCHS = 3  #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}
LORA_RANK = 16  #@param {type:"integer"}
DATA = ["/content/data/router_train.jsonl", "/content/data/router_train_args.jsonl"]
OUT_DIR = "/content/out"
NAME = BASE_MODEL.split("/")[-1] + "-router"

In [ ]:
#@title 2. Install
!pip -q install -U transformers peft trl datasets accelerate sentencepiece gguf
# Colab ships an old torchao that the newest peft refuses to load; it is not needed here.
!pip -q uninstall -y torchao
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
#@title 2b. The training data (embedded, no upload needed)
import base64, gzip, os
os.makedirs("/content/data", exist_ok=True)
A = "..."
B = "H4sIAAAAAAACCu19TXfbSLLlfn5FyBtt/HTscnm6S7PQgUVY4hMJ6pFQuX2mZpEi0yKK+GADoFXsPv3f52SCH+A3ACKRIHU37i6bSIK4iIgbNyIy//3O41HEXnj07vr//vtdP/Bj7sfvrt99dVyXHJ/iIaefzJ3wiH4EofzPOAhc+uNdZ8x9Yj6x8fiPd/QjDDz5r5OIh5cRhfyfEx7FV3QbjKfyH16DcBAtPkIRcwb/hwYB+UFMjv+T+zExfxoPHf/lip4iLtbm3jieUhSHjv8iv3/+ieU6A2cgl4jY9OoP/w//d3mz13/4/yXu7Fp+kI3HFAcUjLn/nlhEjMY8jAKfXoOJOyCfeZyc+N37d2Hg8nfX76JpFHPv3X/ep5+Iz/mABk7UD0Lxja/Lj4v7WPvwv/94J5/L9R/vZpf88e4/yytYFDlRzPz43X/+33/+178BQlYQopjFIU3GNGaOH2eFQH4YAJQBgMsmfn9IP91+1of/0+3j0Zf07ocx3n19AAQj4oMXTqNh4AZZn7+4Ao+/1Pc/5i5/CZmXFYL55wFDGTA06Zth2WR36Klnkm30HqhtWMad2c2MB4tG5DGfvfAQmJTGTcfBKw/HgePHgp4Si9g4c5BYXAo8VOQKNHY5izhSBg1YhBOf+sMw8DI//uTTePplPP2+/Ow0mMh1RN4g7OECyYNeHNIhGGFbFygOvTI/FutMIo6cWlNwQDpRi6wOMVo7BEKrgK6hNwr4QczHbJAVhtnHgURpadtrzvrOKzK1svzQOIidH9N8+ursIiBQFh1CENAfBGYvNWygXpkyjd0I2XJ9pCSkyxriw7JWgOqC1h6MPnP7E5fFQWYFb3kFgCgtX5AtATnyBXCmku0A7TB6Hv+QT0kEhIjHseO/ZGZG888DhRKNgP/F+25mF/RXn7t4/GU8/taTdXtPj0bTsuGCKn/6U8ZCWsjXUO505gX/CoLMJTTxWTz58i0AObF2WULGVgTi6osHAaF+UBMyOu98RNej9rRAiBP5YIA+oWJgJ/aRHmh2R1dwRBowWKmf5YrMqKAppqrI2PTPT5n/uDVbSBf05M1Ctpg5e8QGPQO1KznbBZI2TeIdajj1af5aVulzzXCiuK9W10NbpK4Oi1mPe4FBWnTHl4jF85A5kjK95uBLqLGpCRIYWtOJxcSThoBym/awPC81xEPmjyJkcfpU1nmURoTWV3CYZWU5K6DI5cpmrPNom6cbCRFaCVlNZdM5pneQTCuJEaKKcIOSg9a55gukbpo5K8RWzY1iNZwk7HEW9ofJA+bPNQXinxMeTq/pdcikDhGt3LNY7iA/Xb+AxtN4GPjkOlFM/cAbh3zI/cgJDrbLyHtJGmZ2LQGkCiPlBsEo6VyKYkYh7ztjHlHgu47PcyCTvhpoHI/GNB76wZY3fYZMFoYLu6naw6WNALajy3aeeRSTy8ZxMI7olw+//O/c3mxjBaByNCrD4FVcHzucmPwzLyjrCwCTStgZIo1utG7vTeOR7ju22epR06K7jkE90+je3tOD0e1kB6c/5EwYYszdSDzXl4ABljKNyGWxiBzOeBgIihY6/Rz+bcvFQKcwOo9Gzzaoa942H81eMXMBLyuXAWxxP7lJAFxYmci8BMGLyzeoGThZPcJJHDLxXBxPVnrHE5/TgLtDJzs+u1YATsfazEaKiOxSZ2zZ9aLPAswFTKZerm1LHAcJ0IPOTtOZrTRiYQDz0QjQamUmhcoNcpnK481mkl7MTpDtKyBl0F/qhgiK/rVER85Ep2vN26X/Wa05Q2sr4NLj3jL24MPLqU0/tyEjjIf7QEYPMq1O54GeRNnsm9iAw26aZMg/O1araZmQOHXoAAFl1GuQb2qE6alNd53OXcukx+/2fceiVrNn022n/dg1702r1+xYoANaBYGdhK1QKgqIFIQfwzZ7NjUf7zuWSY/d5q1ZrAgNVqCASO+MPll2YUHsUVoqWK8UoDhdA1i+G0aX5pR6JyvIza0ReSoptm02ROdSdFCvLg+dYERoiD6hEhsqbFVDsWEWhdCAbVTFmW/Al+sV7UGY62M0aHOqT1zhU8rCjcGLaxZkXD508iUsiDPqK9B+gAabWnSiDcfoRKtX8QyjZ3UfcLoCJ9bVdPYSBO4LiHENOzWxeYlOFDY1+kLBHDlJeZBMPDrU3+eD/tZ/XDZ3jyzyRxWtfwd0ZMBT302aclsQWHOl9Xzwgzo2/4Mc6EJm204zheg0Ms2ySzCYNq9/hxK2OKnrlsD7xjTRdql7tgl7A5y6UpCcLVbb3YKMwUA+T/GYa4qWgCk5wkHeJcVO7PLkH7h/TQM2JeYPxBPj8mAHcUvXFPj0I3TkP8b069h7T3HgBWEYvIq/+PjhkHgqXKT4zgH3AnnPvx0+L1HcaYLh7MI/3r0Xf8t9+be/qTqfCSBuB5ENBvIr+2JsOhanObnC3r2p+L/cH7CQ/ODwxhErsKbWWkVXrAR4q4Q36g/5YCIEQGlukfPiUzT1++QFvlj044fcJrtYYxXb1IKAuEqIe7f3ZuOpZZJ9b1Lvsdm1bHpsGZbVtO7I7rQ73W7nGxk2fXps54I6GoeOH9PYZb7v+C+rcKdv8dMYkGtx2msIHe249yIO562NYD1PBi88ppD/dPirvPnP43xue2WFVVw/w3qrhfWLmOhMnHW3mXbWXztdOOw3kiqlVjsyZUqtBFArBVUcgbtCsGPOPBLPfDAZF0Y4vQiQrhXfWsF3jWwVTah2w42MShPc/ZCzmC8ezV7TlpuX3MDAzyQ+j8PgT96PaeT0R8GPH8nzC3znZRiLhf6eC+i1xdb52HJVgK3bvCUvW6PIR/IyEO4aROy05LkWsP92/enDuCTxM1kLENcM4uMMegfWsOv61DPSTtsLQqF6lOW1Z8sBXt2heYOSpch30jlQHikDAddNwNdLGkXE7r06KORunS573Zhzl6j2mi9KVPXLpRCWTx/l265p2CYZFpm/m5Yti1cNs9e8s6j33bqlzzkLVTu5NZxzDex3tdqct6dvT6UZzX06Q+86syqgf+wlVtBAatm9WbQutbOJE2WpOhLpojDv5dOAum5tu78x76IcqoVgrEvi+OeEhTEP3Wm6oe8YqXp9QejV9TRktAe9QfHySLl6L+UG4nUw6zT5/q0slo3wrMuI03CuROYiSuZOfCFo1qIFv2G2O0f33aP3p57gPnY7/23e2vTQvH3ofP0qcRYqVU5496bIUL3qKFoX0Db36NbAuJbKZgGQd4ZjQKyBRAe0vXy8lLuKT0ntUL2QMdXIZ28KYQXh3i+AAfMaYZ7259hY5MztOT3jUkT8OiRsY4KmBvLXKvM+po6xh4Ijoa4BF19H+lg2jpTrNBRQiNpnWZkq2jKytySFlpEax+RZreGm9KiMIobWAUjZb7998hHt9+fcROJhz5EzwVgelLyr6+9zWbPLmK/QuTnnej2y3bEaxnf6+MFoo2f3jfXiZzzcCS35pzn+mOgXF0fSLqggNYL089i7OhJPhN/aMOffmFfWrm1QteojdWBf5JPuB9nc3qfQLl2Io6exlU9ea92b+MBe66Q4F9wyc6/YjHJ/jeu/RXu39iCOxq0aVRYWhYWClr2tqACDrt3BImuNNuQHr+jheSN0rADX1jYb0+B9cfyq+EGO/9OJeU3Blk87QTu5T+FPB8nNv6fnKTlxNINaYCt+pHgdkitmgCfGtAT8MpoLEe/lx34de+KXjaJhMD7wGjjUZ/5lTB4brfA2USp6P78vcuJDL8HMvlaNPqk3Ae3aoD3H05vSr9efPmQHdXYBwKwRmCGXnnZ9pfGsknSTw2Q3bgZA19BqxWUfP1Jw2XeD/kiW/fK55bWLAXL9rBme+Qyw3FyCRpxcZ8rJYz4T59DTIICDflt4A+4zgTtafPmivyf2k/6eQDzuSRgN2DRfaE43CaUWAe51wp1NyZf6Z7L2ZbScn/A4jzO0VS9B37kEIK8j9d5YCt78XIL3JPRpELz66TtabtQHIz/7N8CbysrGFspGYzfKlYp9grxdS41sN8MCTTs3zL8bRpfsp65Fjc43Sw5aNK3fm7aZ7Ot3bwodrHN52+rcPtCt0WpBRDtDc5+NPCWL50N45VLgWyN8d/Kp4/QW0LTTSr8Rzt9SfjaftXAnfn94pLC6shZgrhHMPeM7WR2yO9T+nmXDZaRdNQbzecicFY0cbOzM+xJSHV9oFTsHcKeMiZ+dplzLyImIey7mC+s9e9eMTOkNZUpr6uTx6RLkzpOQOxGbz9J/7xY8s20XBI3ztHAXu75tjOQsbbT4RA7s/ATsPLrcsM4bWPj5kbQVSet4igaFrP4lq0SkvoCqfRYliuSkyF7zzqKu+XvT/AaDPQ+IG+Ztq2mZAPjcNZL+JTSSt93enQRYBOSz6QhCjfktmS9Ez7dDvsjoGY+w6Ddl0cQiNoZZn12URqvBWe8gtmW9QiULzMyeSOEZGxKdcSlyuSnRp7F3g1z5lMXrDnVNedoUJiDPXN9MMaXjtU3QrpNqvp/1BV0B5PNLnSB3vYW9hSYD7C30FopP2ObzvLzzeud9rq2DQKtPSORKsttiwhYy49MqSKCo/Pa2/MOOf29Xv0bYfjOuHXOxOD+D4iHzR7D4s8G6SbeGdWlT23jY0kj0nuY9Rk0bZO58Kh47vPDx1Q+495PYvllIY8eBDX2t1vraCWy/3+MxTcbElglf3TGe3WgK2djlCare7PzR4Fr8ISBIXohDMpnfdycDTiEbThaPa+1I0HwOePt5ooH8d/ktAFMVmOLQ55/OKGSeOHSW+cTimPsDzpNIuBoc83frpq5eQTX5SsCqzkalZ/c4D9na6Ez+ptzUhSsgytWBoWIMR+zZCdcaq2ebCuTdHmbXUd0JnPKLAKdKT5vY43ZH+88JC2MeutNCdYj1q2Go1SHbtG5bTw2THowvzS41k9Moeo/dpmXTY8uwrKZ1lw/NaBw6frwoL8BMqzXThNluN9M1aC7KAxZMVymwcq9bgW60k+uuxMaywmoEnqvPWlP5RzmJC4xUKZ5yT/kZ8fX5UF3uIhYHjMp97Z/B0M8WSMuLo+I7Aa3irDRxutuy0rLiJjyt4qRFnpb3e/Oha7Tn253dtpqmZReYFoPOp1eLT0Si2eMacC/35pJegCRTs2Lw5alxZ9rz3VAeWz0Ieydlg5Kvzp6WnMbKX8VcuxKMtTbq7LrwHpVlnNBmleI68ZCHvB3mo6wLASFUuffNbqIF+hBgp1obTPhgB7IlpCo/+QDoKVZ7RMlkpvUoc7Gok1QQLYUlbklRiu7DuC1PgT0q9qbj0Jkydf5ULg8ENVEdJVI6GI5Omyy9Bg0LrZjzKJMIwHnU4yidrao2AnjWKlgr852YlVjdkusBNsWGl+7fUdjcjLpINe0e7mTpRb3g5kgTRHdHNfl+MtrlbEkuLsrKLtDjoT6/2N2zrKDSDFZagWVKWloepQERrQK0aOlL16TsQrN4kMM1utSkcFz5LB4qyVUXOhSCiVpHxZ3mmOc5SQwTeTy9w325jBWKeNVKqgIQQWEr6pdbtMul93Ypwl93OVVwHL179iirV0HqqT5cKqSvCJsaRFgFkRNmWQ2QyYCIo6itA5MgGqwxVdi/KqclAMZYyeBy2zS7xnxu2TaNNvVsw2o8PZbkU2GNVaD43517S9nwORLKCrjqzJ2q7pSDV1U+J7mV5SgDFP5Va1OysjEe5JQ6xwfW0/2rEtUChFOVwPY6ZDQaZJn3Bhk9MiwybNu0GqZJHUsVyUXvpN5hrZK70eF7te4RoiymgitVL+8BzDOa+kF7Ze0dazLjs+JYlftVzAFVfgiJguoXHKrWXtl67zbZDn7yE8RPoOMFP1cPZEudxTY/nX7XAWyrJ1n6/0WxRJ2H4iB7egl4dC0OaVsu8H5+tJsXhL58gXz+V0yvnI/eix84DCYhuSzm4YHXpc98euXiee08JXvtq8jxo5izQSnnb78XZ8GlPjT/ErxdZ/F2BaMtN0WjYHWR2cGQ+c/93XKK4PKFWvkKvE9n8T4NuZA6o/6QDyYun92o+DmfPpDn+JOYR8RZ6DrplTIcM/pp7KVfnc3V8P6cxfsT8vmrs8UtyV/li69hMX38kL/Fdq87Si+Nt+ks3ia70+50u51vl725zk8PHeGK2k3ryTZ7ZBrdVtPs0oPRpUanrDcJ3ulM3ydx1/IWPn6k4LLvBv1R0uwfB2tfkS+BW1sOrPvN5HSpnSBE9Wm5YP4kbm2p9Eu0WBXvz1m8P9HQ+RFv3sNyuULlz1SLAt6bs3xvxpNouEdGWuPXJetH4Ndn7IfSk49Sx56EkfiGY9L+PeuCbb8Vtr3XW+1YrRR3BXXyHNWlhTK5qIC6E78/XLTbM48nP2f+3QVqqnLF9Ku0fWW8U+fDqTbfpl/HXhmvzq9jD+/JmeX83pR+vf704eiq7WwdqEVvngcdw7IzkSFw7DN9t7YX/9c8U1L9L6s6Ah91hu9R77751ab2d/r0KA6/Tspux1T4wZLfmPJ4bPqVKY4hE3s72f0uSbJIcpZBhkSmdpaZmmhfW2k6KpKmrcU2VELO7p3pmr3be7Px1DLlaPGXp8adaVPvu3UrdmNpd6yG8Z0MO1dVbU9BH2/QW9ERixVkdwuKeHPeypvzuYzX5TPekXPkxqttQ3GAziG8P4d9zLxwsVJ552HusaAthQtU28+4W3rD2xzTK73T5UBXfuuazg1EHbxIh1vutw2WJX9VoOl+b6kLAvMZtrvORMHPxyiASKvOcZBnleesTtAXGufZyXVAl99im8/nkquh8ELn74U2O7fmruiqJF+E3rA3kHtt3YzhqOEeTNC/0f2G9mdiWxt0Ss/L0AZ0vq/Yl3ujSal6/Ndus2F8v+zRY8uwrKZ1J466scX/2h2yzH/Y9M00H0qmVqh/nGN/q2jtaJi95p1FXfP3pvktb5F1J4kCFT8TiWg+EJ2qxX+eFcguUJDHi5JJjz52MgwDGG+2QD/bbrFohXVTqsbbco5jX7L8ubWUOtqW1+cfAcPeHdCO9mlHhVtAENrebH/rRh9Rwf1fUFpDF9GOLqKVUR8/7/YvGSIe+u/f0G6eJbQA7BgAgpc6wwaAJHsbrXmhhHrfYOAQL0uxjTuIRWwMho2XKefLtMUL4awFvEPZTn45wLMXlf2xG5XOsVHQP7/GyNTs2ZFHCa2PnkGDPPf5M+zzirepPBVyy9kvufcT2nPgC0bO3sbIGRod8X4d9351Hqjd+T3ZcMg2jTa1nqzb++P2G8KmMW8qnKGtH29TWf0kq9XZHUcIo0iLd2jPO5TK1EaSVhd6bXCUwls7SqE4f94iCIEmn9vI2fKcsnRbPtrx8ZJszG3sqll8Lr1KgVfnDRwqtsqBcx9Yv9PvVECAH4NIwE82Z15U0/foORhMr9PfHdA4iGLxT/0h833uJv8qf8P8r8hnHj9UO0ieWiLTxaw/Cn7y8IcbvC7XdS49CifJBKiA4RCs4l4ldusXSlxny8oPrHwhwC0d3JgL2X7IyXX8ER84/uJi4arlavGQzby3x+mZ9UeZ0U1ftIHs/AsBaumgjmc/Xlwc8sHAiRdXSzCTGczFI5uMB3lsduvVG/Am3wtwSwc3GJHHFw553WyvYaknFlvnSL448XDyvFwv4pymwURUGj5nxjF90QaOyTcARaX+diOOzjwupx+Oy8kRj4INpjm87dqVME891PfVGTnjAXeYShOVX8IHDgOY6sEUz1ll3ASY1bjcF485bnX+Vn4dIC0dUttstWQDR6tpPZiNpkW394ZlmS3qWLKpo0f2vWEvtqwSDR8N0eKR7GqVGeG5yiYktsFy/hpxVYPqsGq6q5KDc+m6FHF/IFTLOPCdl2GcQ1javBiWXDNRqQodAlasMe3Z4Wrhqk/Qljcos3KFGPy5Alh9Hv9wnb8qA3X2fYBUaX77wrwlr7quzA+DUKnmyysisQY6BQ25GkHDdSBovJ00aA2hC1QHzkBQVtcUA2KsGFS+jRqrqPeADVfChtdK6+VnNaBFimAUW6nsg3Ira82yWyF4b81I0tbW0jWmVFUei65TtaC3zV7PuEvGte/aRrM1L/VdU/OyTd2n5HCalmGb5bEmpDnqM9ftRqw8g4W9qoW480CL6vxd075/+qK5No9wXIFn/tZ8aD6ajaaR9s6tFvVMq0FNm+yO1by7t8utySO3VQfuY6dnU9OS4H7vPNlPX8yFHUvzvTVaLWqb9MW4fSgnMZoGk3jyzAGm0rC7XsSrqH8G6oUqcKeMhYTBO7BmQYiQ8J6VGsk89q/APyBhlSxfJd8JYEsHNgpW3PRGyUCdg0bwrVXLW8YNDdD5phvdiUe5NOd0/e6mnKofom2Nagrq0yTAXRHcq8xKvR2DVVVAl2cK0uqKK+WAqxIrCdCrKmLKOwQNVHjf1ITfao5zhWLveXlmOOYTgPR5yBzKZK+VDBjAcJUb7uqeYGrtFvuAac16quJTSIWqmAtanf7ayqGynCKYg0ahz0YXrJvCUl6JOdseGEBYzyac29C5KhdbxF6t87gVbGAD41VvvJtVwO2BN8vR4pjxq9VOY+tNkKvNzOqbXSFqaDRgbFhzqoCuZbwb5T1U906xGY77FdZ/AK5qarxWHcjAjXON7SK+1todlx9cYbGKoPxybzQpPRq0MfeV0GH73rAeevS105Wfenps5BnPzVb9AYNSu5VC/lYpcKmTHSyADz6ndFVlkVax2zVlTwGTN1FTZMWtpQrh4rjCOGb94fL4xED8JZe/RDLOA4gmnxQLPk8GLzymaMhj+cKEzpQdwi55UjPslgvwOH3woVwJiJWLmB/EPLqK/5JY/RkM/cxQLa5MYyRWAERlQWTKvWJaRs+mb6b5cNmjrvnY6Qp1lizz3siMlcui5HxT+aTGQbgCms+HsKvSQIvmK8hFHf9n4PTFfQ5ouQSFbDhxc3nF2UJp3OQiAK4s4FKoRa4z4NE6aMx34nyhLFln9fReJ4axlWdsQhFL21kckHBnJHrdE+yydZEcsjX4SCWwib8KWX/BPpa45QJsvgyoiHLEIscVrnHDzo70i7AvBaHMm1LIo4m3QT9+8kFmvBaLpOH6yQdAq3zGuMnTj+Mg+3k/uIgio+v/XIctYl4eg+v/TOMUMQ8oKWD5aaFpa4J2VYpuhTRNkdIYs78o5PEk9Od0JBdey8tBRBRb2+G4loeR7I9qoCalG9uSR0pBP4eev5U8Qsgvmz4mH95iZbHYhYCHbDbLWpKJySUBoOqI9pMPrkqIaHCIZesg4gm++B7346tB0P9rUeYsoIasLQU3WTEVCQNQkROyvO0hrqByvB886MfKFX8pQJUo+UPQ0pO8jdizE5ZkdnItYKiyQLrGVbJsoXCoPAquoga1FYky4SYF/eUudRIcRUkmlw5z+VsgtwU3mJiK4Laeyq1Ftlyi8p5cDqpymdjJ40J2+khZbynZS6KGo8L4ls3HR1W5t3YiIxeoKvsu2AeEnuSK0UsaEzbT7rwdkhv9CTA1Ra12SRfykUIJepArH33is9mnn84oZF7e2tvebE2uCOzKxy6lj+STs3YoItCxVLVtLVLrLbna1dFpNoiHctl/RcQ6HjEoWCUCJndHX3T9JHwxnz/coIfwhKrndvN0F29NltFhXKG4Ibunbo4DDB1YqtjFjoHdhHgfSwpB38vFTe7lZNj2rXEvN2myjX9Q17SfuhYZVoPk+LzczJS6xv1Tq4RmOgjB6vlh0rFaXlsInKUazFa7VSXNK4jadlsDcaxKwi/avbpfw0fFuoLmgtxjTxA99G9Kgd0oaofXxkZYBdDCllgVOcHdbeBo/j7lRuKkzpnlZBFscVDTKkySvGXY2hxpW7XYTbzZh7dMr+XynHvaHOE21XXtrOXX+bwkijLVQLUcky/ahbp1Yh6qo4J9VdvfqWv2ntqm0Id7Rvs4iFBCq5Zn5N+MExODuqTh4lNL2NFA67ZluXjhBscAG1S9wz5fqlW56OC+FlOgppYYFq2TbWUdoPGqW++PmSbbbL0HkVffPhAxr8TmAWW8/qsjdhI6IbjEuT4UhPQqDzSMyYkOoNPq3Bq2ucyzirg6PPkiTz7iLEw43bgvv31RmSpSywIGRTD44WzuFHZTXGQFCEVAcIO+OKt+4VBu4IMqfvSpDoaLYm0PePzlPH48fS1PfylhFVa+gEERDF6HXJ4BuRZKEYIrhyGUOKQzrIJpGRA40hmlRb8j1ELgUEpWhuCsBYelHDtiQ+bTkDlICypNi2fiKI8Kyal48EdEgRVW81cfbKhaFJZt5gvfk3f/JShzR2LQ69DXptWQE6Nfnhp3pk29e9O0QYh0pQZJ84EfvF7k7TbAgy/y4Id8SouHv9lBf3FU9z0gOcoWUKqpS4J8hYCgUysCFDqhWOmqWHLVAnsZQr4rzTvN2pLyTlchcS7x+UOzqEE8QDjQIdstxE/IpVprBigp6woFmxPs+UsIyJlLryUsiGrxuXMgUAoCePqa9FME5mqfPSoItaVH8EV1UCmQJuuzgk2KCXaqRUFNzcHnTxSQs5VfZEZ9udJnvvAm0SVqy3qzNMSEugnaaP2t/Pm3v9Pt7/Rg3BsW3RtNRINKRSJUcDSi8JyU75e7UhcjpHBAx6Owka9Bv9MDxLaeeEwIapcvYA5axAoEhroq2ReQsiuGYqs7KnBSOrxSSUUFFHV0PP9+MNs+B8qpBoEoqYrdoJKmuesdPkevBcAAtMwns2G+tx9zaOXQHUwA6nM6kIJqVx/Lv7U5pIjSdy6FIWhqL+21mg2zVxsG1AjZj5iYaANwpzV99s/BYHqd+u7gh/yPgbj1la3G538pHM6PIDwkCwWzT8sblIcxzLeHj9hU3Jxz6bok9ySXm5f7zsvwYAwXNyuR23ZxRVvHv2FQX0NHFIBml4i7+skH19Rnrksep2fWH11lhjB9VSXn1bxh4KaMhbSJnjxz/HqrJdLYjUqzRoVnm79hUB+75qPRNckgeUKb3aGucf/Uoi9PNjU61qVNPdNqXFPzsk3dJ8tqWnfUMmwzB6wehRPfF7/TZTGHg1UNaTDaYqXyUV/PNpNYPKzJeMBinhnLrVcDUNWAjkM+ZqHAc5UDPU9iGgT+ZSzd5rU4CDgWt+oF4vi3OKAfoTNg00zZ3ALjHYsAZU28aJHQOOLXs8E0h7WuXQmCpBbBzVxFnhM3S1UUu151h8iB8s4Mc8UuI85pGkyIxfQ5e76SvgrmWHXYlOa4HjXXCWopzBbWWBWx3TBG2OJJkBt5jmbJyCk8mxPYbWLnbU83sm1YmCfhAK7V0NUVaT0trR6txyJlrAZBnw9ZRQmH+CogqlyOfTC+NLsbcqx9b1gPPfra6cqS5dNjI48mexBbOFwV4E68LeXMn84oZN7cZquS8JJvBcLKU87k7Pn1nFORqAezrYb/iuz+GoLeWWApaIwygwRHqgTDWQtCWha4OFpMQL9BRVUuSUUU6bDgORUlngnPmbFYiHr1t0Mn3NUhshbP8uUee8MhVKBqkg6pAmXsDikzqwTdqa4vpAI4UQirwlhnAtBmWVphZztoUUWmOguqGFM4F7eLToOa4/at27RF/aTRNb7askDy2G1+N0QHe6slqyfUtMnuWM27exvDCSepvFbAfaC+Vl6vVhokkZoowDPxsYZF5mq1umd8F3NCPdOk750nMmzIQKfW26XUGIFlNd51tbcAGeU5yAXRFq1gU7NNqss35ci2EYCtjNpiGuGE3S0f7PO1N+U5W+gG1XUVlNoHDZ2gGlOUz3nZCK2kuQdYVoMl852YKZlKkCsDwWqbQyoQ75BfVhMhpf0oa56EdVZOWxW3NIO3KhkqgWGeA5DyeKRNKD3Ow7KzELkmANS0TVPMhtiD6/TDZpRW1le1N7Hv4U0pyh2k1ypHEKroqUSdpMLgOZPwIACdj66uYkgIaUm18VJxmomgqWTvrWK63uwEvxvoeyfbvV4BLULSUk0v9IPR/NK9pluj1aK2SV+M24ej1QRYY+U0KA0F+cErzks4pQ2dobGf3jk0m+JeUpbebMVTii6k98qTlQrYDxKWKmYTkoNNZrMJbdO05f92fjcb4l+/dpsN4/uJcNqe6A2cv68ngquwKu4P3kuZLuY+sdT9eJz5sfhPbwXyIPmF7+UH+V/iiGBxrtvB0+T9QTJZNPPWavvA1I0WAebD/GomQyjqnFaY2wDcgycvSEOeDbGsmvLJzrIA9cOChd0hw2raqme2FXJpoLwP5YRlJ80RsTyeVYXrBv3SHZf3nQyZa1MxjbsvAOp9UM9CsqfGgiMEYb3Ua0sOpfyYV2RTelDv3Xc6Nv3efOga7YVs8l6eLvC12TKp2aOuaTS+l3T8ICi2LpwFm06ksNkhEzNFTBHSoGE6VTAhTlYigikrOwHkfSBLA06sGYr2+cI88STLHgWJKY9YGNDIUauCgX1rgVqeUboJ9EntCgKED8tfSe/kVvnrCqWLkyZew0Dekrd4RO/LHd+DZ9aC6+L44Xnzz45un7wHoOps+gHie+VNMSe7EYwraAhSOKALxPP67lOdVgHSB0sXwgBLDc2QQfTJIP/dubdUH4yLnEkPwEm2VKn+gfxJp3CddIesStelnr0BLVMTvCLmVqJvIRZrlLeSDTilvKWYP6P5R2uulOTIy2ypVCeNPFgrtrMgvBXbK0TgExYyE3znrfULgE98AyTAftimNzAv90Q7oKuLVie4rjPr7RWKZE+WEofOAbtOGVP26ZaqY6ILV2fylGwtIJOnlZ0+cpqthm3vgW6GCMwHqfAL6eMMJ5/OZENRoJ0J7dkW3VA+zgFV0Tz90EkmIx6MbocemqqLivDW9eBaSkIxyNYpNG/lGmFEA1fNO0Hk7hGqvTYMuy7bR1TThYtyha6sOausiZ7M85uM2ezRukKH13ngLOy6igYRGHM9wjTa986mOzNx0es7w2D/xXOzXpnfrpehMCRxJnpXqQ4Z6XBNttsr1VTRXKt1PC3p6VG1ZQ8qhzo5lNzoY5VCVVAuxgCbztaPBPNU74fC+TUgXQvmjP6eczPihEFvb6DOVUcEla7fYMTCeMtti4e91mjioaIeAPDr2m19ulopOJGD94D1wYaPntFWe1YIakhawE12tl1vxyz19FoUkmrVn6ckXUI3Xm3Oc2pT98myxD7VrTzdeJo6bB9dNiVvEjn9mkL7OmRxAm0U+C/viYWxE8UUhDRy/IE4S1Pe/QGYWOj86cQUOf7LkFz2woJDwIjvTR5+6lJgcAQGDv3g3CXXGXEaciZvhVEoGig85mfHY3EJ0DgCjX9O+ESQfIrZ1A1Cil6dH3FmENIXAYZjHNPibc7rlmAGZTx/cfC5+KV/sn/9K/OzFx/GUz/iqY8nMQU+Be6AngPXnb4GwUAuGGXGYMu1gOR4SBANdIIgHEvOOABfdOQzl/fO+cgf5Hzyywvx/I94/hOPFlwU6UCdkjNwokoDsPiZCL91M4J+4A4ENJnRmF8AJI5AoklfTbNFreaDSfem0RVy6m2n1XhsHT7vD0iotoko8Di5wQ8nMxSLK4DFEVgEI1pnqxegq5p0U3DVOgUH+3ur06Xet+ZXG7ypakD+58l8EtuPUNdsGf8QcDw2DauTGYiQu+wvcctjh/kBoCg5XC89P4KFFlF19f2GWWgDYkFDL8Bcqxc3UO6peTOGEy46XdAeUwdEEDh0o7L6KHNWiICDqo4BRJAaNZAhvdAEweOTTR2LOq0Gfem0Wt+/dToN6nWsux6soiaJBvIMPU5p8TgzHikCHBSp5Kn0ADmFzt4a0aBxg3YODYmDz5A41EvogFPSz1ptAyU7zYLsyImGQchpNPFYmBmDlasAQsmF7Idm777TNenhqW106bHVAy6V4/I8ZA4hk6tHBrFFociyjw0UDsWDQlD96tT+dG/SN9N8sBpAo+repydT9j6tBG4E7cqBmHd750y10SSuqusMoyw1CxK9TtukVudrE1S2+lwbeYTmPAK9NLqhmDIW0rZQAR2qZikdpop01ktRmdAZqGVCgHlfDU+eof1YewxAKNYLw5BPkw3JlrwTbFULEvPtBqhl3BkdhIOK+5bSSlDejZkgIqmT9baV47IdRI+KXEU7mOXFAzlc2dmDW5f0ocdjsVEF9xx/wMNaP/5XeVB6QM98fr/iySf/zv1r+ed7sZG7uKNreQDR2HtPceAFYRi8kheEYvP094LFzo67YTH9esh25pezmD6NPRpzmjI2oIHjMp/tOkgnuz6y7xQd8YsSA1y9CbwKWl6FcXJAJa2/EnFAW4/GyeFfdx2sgzegVm+ASL6ilYd0vfMorTXE4BDO1yGkrkhOFsl+bt6ymLd+bt4C69TqwFkLzusmv3Gm9Py8LLHY33PR6s2jqVM2vlwTwNcj4s+WSUJ+ziPJs+O++mXAvhZGjzj/Zl+F5KsSR7DzPdi6XBnvAOK/bvw/68r7P8PkNUH+G9uAfNsBqZnh3nm66gLr3xiwrqXEl/c8zhTmW05sRGivKcNfJluK2T2yujrRufShyBsLZ98OY+NsZaRzNQZ9q+BOXuCLxT5+YF6Z0n1qWWCvBftHMdzSNulv158+PLbFYcw906TvnScy7FIkWrEwQngtRbot5nmlpDAH916Hl+BzUo/dlm4Ri9j4psSMDdl5XbTZrXCvf0eJyMPo68fp0tG5kBiLCuwJSDTz2L6m0aTTrwskbWca1UturED81oXrxCP0VCBorwXtwoRtt4nDj9cC83SD3J7IXULgBlWrS8xOyZ5JRq6kigZxte7UfEejg6I+Kfj5GjZEKyumwtvXeiZGdYMsOic0Im93rObdvS1KaH+nR5O+G0aDGs2WYRlll9eQtdWtfL6eqxfRW+HbT67KssrEC7e+ozHu9Gl9yVosCHwtUO+ZNhnUNdtNq2F2r+nWaLVE58wX4/aB0hG/lBoLorpuDx9QptmWQlJ8xsQdL4HmlyAt1FXT+g7R7hQmmotxO+TuJ6TaJH2rezx/uc4ebbJaCX1qwkVtEQ5BXTvaM8tWizPsue49sKnlLkrsf4U2Vz8Ct22i6fO41EkmNMzVfTOSz2VvQALI6wL5yqCqbIjNt4nr7jAOjGu0sYiyqjlQ1oVyIqcLGd3urCrqvxntEuwXO4do7GNf2UAiHi42kNgppefz2hDRz4mP/1buzgIwfF1wy5PqsmFerDXmIPZIwU9iV0DsDvmWh1uwE/TZ2fyhTYQguZ37LIPikx5g5LXdD9J1VHY+IoifxJ6A2F3iXDAPRpRhV0A/CPwy/btYD4DXjanvyt/84BW7uZ+hdpNhF4qcGTsi/AnXU4uCniHGA/STGGHLv0lgphk2MDzNDG/9DTj9bQJjx6st2C73X+LhNQ2DV3ID/+WaPn4gz/EnMY/e05C5P8R3DINJeMhwYxaKH5u+hPrBxI8HwetBPp7chgQlvQBQORIV5rLQE08j8Pna5w8CMb8EIBwJgvyR9Hl+bQ4IFtcAgyMxWDxJGs1+MWDQESWWP1XehXiysAndwWElZvvB6wXCtWZEPhaJFh9hGqUDsXikV3BRGnAQ/y91rbwTj+exiQ/AQplNwCS0mMTEj1dDdm6rQNAuEZBfqN20nmyzRw8G2c222c2BxC+wi1Kz7CUHugFz0ueelg+0QMgGGKWl28Es1/6FvLwRG55JSVbnO8jqdAGRGgJZSlC/fihAZ39FWlEWKD3b6Io9DD9+TnhUj247T5bd6HyzYCU6ADFaRrdNTWuBiNkDEHUQyzeuOggGUrwyc4vZZ28AgCYAUMTWjQDIkl4UfvtAEe8H/qBQHXt5NYAoqd1p+W4XanaCaZSXR8id0KX+Sl87Xbo3Wl/JsOi+89SFNF4PBrv9UmBSqRo1KxoViB+AoOQMO797AgRqPNPHIsQWzQXla+SSowZ9fxCB2OotVsAiapBwgzHpR2LIp7StcRk9ULqz75QcUiT7Rtgoue0jhUfutg+AURoYKVGqQJYHTUpZTxrqqDqjxS9HSbXohyoPDwlHqkewCB4wkJKD97xUmj90o8haaoPazF0t8JDwBINc1gFISrWNI9qZ4abKjuPH7C4BTaQcLJ6HzEkRqgJpBvhUWVhMPEKh9QRKTMsgwiI2BtWtAyhoKdTMdmcSCYpN9aJWYFa6oThOHUHEKAuPpKMQszC6h+yRatTbOd3AO2kARGyYOsPk43Gt6GBTZWqGsY8tWurU1Jb0tC1L2zcoiOsA4su90ZzNaPz2gXrmbcdq9NAkUpeutl8KZOEgV6XF8nnBb05xr2qjSt0OuTh0QTxfzuJhbbEYu6zPr+VH+048XW5XHwcDNl1uT/+e4qET0SvnI+4P3pP4HvlfAzY91NPDvTEPWTwJudw7dTrg7JkNDu6ZKm8tCe3TAQ/FNau70QM5xci9yocVU8gcXzy0UeCOWMwyHRi1RG92VVVnRAG/JX7JAxCHcotn5gb+IPCvsiOXXFDRidzAbQO3z2OJ22DyzJzsqMmPV3TGKkBbHr0WhLzPolje3MznHekkgVlFhpY+Nq8Ug1N+UA5A3MlSvIn3zBwR9LJDmFyDSFc1drKiLe1H5HAem0TMy3Ii5V7jA2pVhrrjnSXwqhKvZ+a/MDcIeXbMFpcAtzpEtpxHee6PcpWc4glMFw9rrqHIcBfl4ShIEWqgXiaSyHESCjCr2G8OuDt08hyFmqIr4lLkdjVRwsYTPwdtEZ+GDqbZYb4EOVSwlwDhTa+rfAmS8s7VEaChtlOhZxQPOyEmAz/wL45jJkCuAuSC0briBfWkzngBrNNlH6jDnYaJyQStiIltpmcAqyLm8bfrTx9mDQoChaNAS9YCdIqh6zyQbbYfza5hP3VNsaFy+6n9xWgeJyEDtirrN1AgTwCxRcdqkaCGdlc9oEUBrcshi8rnsujmBaHv+C9HV0/XFwS8VXpRyJKnWPnO13i+jatA4dKA3jIaFveiO4IivKjO/oUJGhhqD5iogqaaK8cuZ9FRRVQgVrH7XJLQQi0MO/gn2hi0zxksYtrRYRBgVg/mOvWYTY447KgKEQiN1viIwHhak+TluE8gV6X20h9y32cOjd0oO3Kzi4Cb/glyDJCfmKNMhDAU+U4yuN0gutUdurWaesNs3TfRw1JnxL41Wy1q2tQ1mtayDYLsTrvT7Xa+kWHTp8c2JutOrK92zYfOOCNI5mkZ413HKG6JG7VamKHGVA/M8wTrDVIRy1dq3xTDUGivXLu8gnhZ7+4yuMZzqvWUASCqPbXYGiAZP8AEwykZpO+8DGNB6v9eXqqXXhVQVm2Kcs7SC3zRyfDxw5GTlql1gKQWJAtvU7WvKwJJfHVuNuTzut9x3fKo+KkH7Guna94aPZu+drooQJzAgBif0trmRuW1s2CbI31nIxSfcNjaHo8EsWJcxQZIBboBMSJWNVATj7Z1ldWy2NdOLqQo8Lg4dyjw6duQxZExHtcUwOdgML1Of3dAEfcH4p/i4Jpeh4H4q9kDSf+eQ+ND7NkRxyPQq/g8G49pxMKA+sx1yeP0zPqjQwiKW0sATF0kUYyDxJGK7wCQaoEMRrSCJaA8WShj/ldMP51RyDxx1dwyr8m5dF35FYLozLSxzJBuuzgNbfKFwFYtthOPJLwhG07cVXRLNVS5PsCswFA9zkO2bqcehRNfFqZcFvMcNrp6YRpR+T1AVC2iCxbk8yGjiHOaBhMhcn7OjGH6ojR+YkXAVxF8s+hZKoAIkFVAOENuIyMBlKcZHqUjXYmO8ZD5o+Vjm4wHeULk1qvhZ6tFdn7FBosV4Mbl2iqYrEavq9hW4Yd1ggs54eShjbbh6nEeixvzgp9cbPE12+YkM7Q7rk+jGwHaqmIs852YbcbYdamAWMTGpQgN8guBbkWZakKhoP2dXFDlg+rpEh8A1gpy1j+DoV9pziq+EMhqQVYpCQauFVDg5UU75IgK+DBCbiWMWFTadKQ7qL3pFxXFF/5wXE6OeAxsMM0RfteuhOVqgna9GLAD2atyoEUloPI8FkZ64gQ56SHc281yU4rKhE7CCiCV6c4WlUKJkSLZqS6QbpjpPJIqlSlgsxVCvNkAXAnGqNtpbI3ZKOokcJUScMGGK2TDiVSRLtDcHF3WgQihv+paamcTCq3VzEWtWWWpGMIoq3SriVUqZkAwyyohTTKZraDcIHM5TUzHoTNl6brMgKupy8gvAqzVUKKNQbe91dW8qQuqcDXKTDGdegZIii2NEk+8pfNbwYAqnHEN1MFtHUkZz5hFa//JTm1cAdtTZspKexDhlit0ywmeOjoQAXP1qkUFsEK4qABWWYhTuwcaqm+VdoKvahVjdpgdX5QWcCFTVID1l3ujSd/uDbtnPD6SYTVtg5qXbeo+WVbTuqOWYZuYhjwp0Sla3+5ul7KYa9IVw8x14sfr81e73TJmsM5ndEOJ3AgNuQJgk4J7NfoxIm1FRYFdjf7YKfh8NyG4wC4Ep9wdszZ2cVHOxAZstAIs22avZ9yZZJn3BnWsZdJq3xu2+MN66MlTgux7k54eG3kSV2yGWMft2hd7higZs0KFrvqNoJXwXdimvplWUN0zOagGJbgz3siy1EkNqLzVeNuNFohrtD+c+XknpUbTQ+XT/w8c6n0lK6UGAA=="
open("/content/data/router_train.jsonl", "wb").write(gzip.decompress(base64.b64decode(A)))
open("/content/data/router_train_args.jsonl", "wb").write(gzip.decompress(base64.b64decode(B)))
print(sum(1 for _ in open("/content/data/router_train.jsonl")), "pick examples;",
      sum(1 for _ in open("/content/data/router_train_args.jsonl")), "argument examples")

In [ ]:
#@title 3. Load the data (prompt/completion pairs; the loss is only on the answer)
import json, random
from datasets import Dataset
rows = [json.loads(l) for f in DATA for l in open(f, encoding='utf-8') if l.strip()]
random.Random(1).shuffle(rows)
data = [{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in rows]
split = int(len(data) * 0.95)
train_ds, eval_ds = Dataset.from_list(data[:split]), Dataset.from_list(data[split:])
print(len(train_ds), 'train,', len(eval_ds), 'eval')

In [ ]:
#@title 4. Train
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# Progress also goes to /content/train.log, so it can be followed from a terminal:
#   colab ssh, then: tail -F /content/train.log
class FileLog(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        with open('/content/train.log', 'a') as f:
            f.write(f"step {state.global_step}/{state.max_steps}  epoch {(state.epoch or 0):.2f}  {logs}\n")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32)
lora = LoraConfig(r=LORA_RANK, lora_alpha=LORA_RANK*2, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg = SFTConfig(output_dir='/content/run', num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
                per_device_train_batch_size=16, gradient_accumulation_steps=1, logging_steps=5,
                eval_strategy='epoch', save_strategy='no', report_to='none', max_length=512,
                completion_only_loss=True, lr_scheduler_type='cosine', warmup_steps=int(0.05 * EPOCHS * -(-len(train_ds) // 16)),
                bf16=torch.cuda.get_device_capability()[0] >= 8, fp16=False)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tok, peft_config=lora, callbacks=[FileLog()])
trainer.train()

In [ ]:
#@title 5. Quick check on the eval split (exact match of the answer)
model = trainer.model.eval()
ok = 0; n = min(150, len(eval_ds))
for ex in eval_ds.select(range(n)):
    text = tok.apply_chat_template(ex['prompt'], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=12, do_sample=False)
    got = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    ok += got == ex['completion'][0]['content'].strip()
print(f'{ok}/{n} exact matches on held-back training data (the real test is jitmo-llmbench on testset/heldout.txt)')

In [ ]:
#@title 6. Merge the adapter and save the full model
merged = trainer.model.merge_and_unload()
merged.save_pretrained('/content/merged', safe_serialization=True)
tok.save_pretrained('/content/merged')

In [ ]:
#@title 7. Convert to GGUF and quantise to Q4_K_M (builds llama.cpp on the CPU of this machine, a few minutes)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
# The newest transformers no longer saves tokenizer.model, and the Gemma converter needs it: take it from the base model.
from huggingface_hub import hf_hub_download; hf_hub_download(BASE_MODEL, "tokenizer.model", local_dir="/content/merged")
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/{NAME}-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/{NAME}-f16.gguf /content/{NAME}-Q4_K_M.gguf Q4_K_M
!cp /content/{NAME}-Q4_K_M.gguf {OUT_DIR}/
!ls -l {OUT_DIR}
from google.colab import files
files.download(f"{OUT_DIR}/{NAME}-Q4_K_M.gguf")

Done. The browser downloads `<name>-Q4_K_M.gguf`; move it to `V:/jitmo-sandbox/models/` and give the path to the coding agent. It is tested with `jitmo-llmbench` on `testset/heldout.txt`, which this notebook never saw.